Querying star schema with dynamic partition pruning
https://www.mssqltips.com/sqlservertip/6984/querying-star-schemas-in-databricks-with-dynamic-partition-pruning/

In [0]:
import pyspark
from pyspark.sql import functions as F

data_path = "/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-*"

schema_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-02.csv.gz")
)

df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schema_df.schema)
    .load(data_path)
)

nyctaxi_df = (
    df.withColumn("Year", F.year(F.col("tpep_pickup_datetime")))
    .withColumn("Year_Month", F.date_format(F.col("tpep_pickup_datetime"), "yyyyMM"))
    .withColumn("Year_Month_Day", F.date_format(F.col("tpep_pickup_datetime"), "yyyyMMdd"))
)


In [0]:
(
    nyctaxi_df.write.format("delta")/
    .mode("overwrite")/
    .partitionBy("Year", "Year_Month", "Year_Month_Day")/
    .save("/mnt/raw/delta/Factnyctaxi")
)


In [0]:
%sql
CREATE TABLE Factnyctaxi
USING DELTA 
LOCATION '/mnt/raw/delta/Factnyctaxi'

dim_date_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/mnt/raw/dimdates.csv")
)

In [0]:
(
    dim_date_df.write.format("delta")
    .mode("overwrite")
    .save("/mnt/raw/delta/DimDate")
)

In [0]:
%sql
CREATE TABLE DimDate
USING DELTA 
LOCATION '/mnt/raw/delta/DimDate'

In [0]:
%sql
SELECT * FROM Factnyctaxi F
INNER JOIN DimDate D
ON F.Year_Month_Day = D.DateNum

In [0]:
%sql
SELECT * FROM Factnyctaxi F
INNER JOIN DimDate D
ON F.Year_Month_Day = D.DateNum
WHERE D.Calendar_Quarter = 'Qtr 1' AND D.DayName = 'Friday'